# Ringkasan evaluasi summarization per model

In [45]:
import os
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Jalankan Jupyter dari root proyek atau subfoldernya.")
load_dotenv(PROJECT_ROOT / ".env")

True

## Sumber dan filter

In [46]:
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = "summarization-quality-benchmark"
BENCHMARK_GROUP = "summeval-v1"
COMPLETE_ONLY = True
PROTOCOL_SHA256 = None
METRIC_ID = None
CONFIG_SHA256 = None

mlflow.set_tracking_uri(TRACKING_URI)

In [47]:
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    raise RuntimeError(f"Eksperimen {EXPERIMENT_NAME!r} tidak ditemukan di {TRACKING_URI}.")

runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    filter_string="tags.run_type = 'evaluation'",
    order_by=["attributes.start_time DESC", "attributes.run_id ASC"],
    output_format="pandas",
)

column_names = {
    "params.model": "model",
    "params.provider": "provider",
    "tags.metric_name": "metric",
    "metrics.score_mean": "score_mean",
    "metrics.score_median": "score_median",
    "metrics.coverage": "coverage",
    "metrics.scored_articles": "scored_articles",
    "tags.benchmark_group": "benchmark_group",
    "tags.protocol_sha256": "protocol_sha256",
    "tags.metric_id": "metric_id",
    "tags.config_sha256": "config_sha256",
    "tags.generation_id": "generation_id",
    "params.repeat_index": "repeat_index",
    "run_id": "run_id",
    "status": "status",
    "start_time": "start_time",
}
evaluation_df = runs_df.reindex(columns=list(column_names)).rename(columns=column_names)
for column in ["score_mean", "score_median", "coverage", "scored_articles"]:
    evaluation_df[column] = pd.to_numeric(evaluation_df[column], errors="coerce")

for column, value in {
    "benchmark_group": BENCHMARK_GROUP,
    "protocol_sha256": PROTOCOL_SHA256,
    "metric_id": METRIC_ID,
    "config_sha256": CONFIG_SHA256,
}.items():
    if value is not None:
        evaluation_df = evaluation_df.loc[evaluation_df[column].eq(value)]

# Data semua run tetap tersedia, termasuk run parsial atau yang belum punya skor.
evaluation_df = evaluation_df.reset_index(drop=True)
print(f"Ditemukan {len(evaluation_df)} run evaluasi dari {evaluation_df['model'].nunique()} model.")

Ditemukan 32 run evaluasi dari 6 model.


In [48]:
SPEED_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "summarization-speed-benchmark")
SPEED_BENCHMARK_GROUP = "walden-v1"
SPEED_DATASET_SHA256 = None
SPEED_PROMPT_SHA256 = None
SPEED_CONFIG_SHA256 = None
SPEED_TESTING = None

if mlflow.get_experiment_by_name(SPEED_EXPERIMENT_NAME) is None:
    raise RuntimeError(f"Eksperimen {SPEED_EXPERIMENT_NAME!r} tidak ditemukan di {TRACKING_URI}.")

speed_runs_df = mlflow.search_runs(
    experiment_names=[SPEED_EXPERIMENT_NAME],
    filter_string="tags.run_type = 'benchmark' AND attributes.status = 'FINISHED'",
    order_by=["attributes.start_time DESC", "attributes.run_id ASC"],
    output_format="pandas",
)

speed_column_names = {
    "params.model": "model",
    "params.provider": "provider",
    "metrics.summarization_minutes": "summarization_minutes",
    "metrics.prefill_tokens_per_second": "prefill_tokens_per_second",
    "metrics.generation_tokens_per_second": "generation_tokens_per_second",
    "tags.benchmark_group": "benchmark_group",
    "params.dataset_sha256": "dataset_sha256",
    "params.prompt_sha256": "prompt_sha256",
    "tags.config_sha256": "config_sha256",
    "params.testing": "testing",
    "run_id": "run_id",
    "start_time": "start_time",
}
speed_df = speed_runs_df.reindex(columns=list(speed_column_names)).rename(columns=speed_column_names)

print(f"Ditemukan {len(speed_df)} run speed sukses dari {speed_df['model'].nunique()} model.")

Ditemukan 5 run speed sukses dari 5 model.


## Speed

Durasi dalam menit, prefill dan generation dalam token/detik. Semua angka ditampilkan sebagai bilangan bulat; data asli tetap presisi.


In [49]:
speed_metrics = [
    "summarization_minutes", "prefill_tokens_per_second", "generation_tokens_per_second",
]
for column in speed_metrics:
    speed_df[column] = pd.to_numeric(speed_df[column], errors="coerce")

for column, value in {
    "benchmark_group": SPEED_BENCHMARK_GROUP,
    "dataset_sha256": SPEED_DATASET_SHA256,
    "prompt_sha256": SPEED_PROMPT_SHA256,
    "config_sha256": SPEED_CONFIG_SHA256,
}.items():
    if value is not None:
        speed_df = speed_df.loc[speed_df[column].eq(value)]
if SPEED_TESTING is not None:
    speed_df = speed_df.loc[
        speed_df["testing"].astype("string").str.lower().eq(str(SPEED_TESTING).lower())
    ]
speed_df = speed_df.reset_index(drop=True)

selected_speed_runs_df = (
    speed_df.dropna(subset=["model", "summarization_minutes"])
    .sort_values(["start_time", "run_id"], ascending=[False, True])
    .groupby("model", sort=False, as_index=False)
    .head(1)
    .sort_values(["summarization_minutes", "model"], kind="stable")
    .reset_index(drop=True)
)
model_speed_df = selected_speed_runs_df.set_index("model")[speed_metrics]


**Notes:**

- **Summarization Time** = Total waktu merangkum semua chapter dan membuat rangkuman akhir, dalam menit. Lebih kecil lebih cepat.
- **Prefill TPS** = Kecepatan model membaca dan memproses input sebelum mulai menjawab, dalam token/detik. Lebih tinggi biasanya memperpendek waktu tunggu awal untuk input yang sama.
- **Generation TPS** = Kecepatan model menghasilkan lanjutan jawaban token demi token setelah input diproses, dalam token/detik. Lebih tinggi berarti jawaban lebih cepat.


## Kualitas




In [50]:
eligible_df = evaluation_df.dropna(subset=["model", "metric", "score_mean", "score_median"])
if COMPLETE_ONLY:
    eligible_df = eligible_df.loc[
        eligible_df["status"].eq("FINISHED") & eligible_df["coverage"].eq(1.0)
    ]

selected_runs_df = (
    eligible_df.sort_values(["start_time", "run_id"], ascending=[False, True])
    .groupby(["model", "metric"], sort=False, as_index=False)
    .head(1)
    .sort_values(["score_mean", "model", "metric"], ascending=[False, True, True])
    .reset_index(drop=True)
)
model_metrics_df = (
    selected_runs_df.pivot(
        index="model", columns="metric", values=["score_mean", "score_median"],
    )
    .swaplevel(0, 1, axis="columns")
)
available_metrics = set(selected_runs_df["metric"])
metric_priority = ["hhem_consistency", "bertscore_f1"]
metric_order = [metric for metric in metric_priority if metric in available_metrics]
metric_order += sorted(available_metrics.difference(metric_priority))
model_metrics_df = model_metrics_df.reindex(
    columns=pd.MultiIndex.from_product(
        [metric_order, ["score_mean", "score_median"]], names=["metric", "statistic"],
    ),
)
if metric_order:
    model_metrics_df = model_metrics_df.sort_values(
        by=[(metric, "score_mean") for metric in metric_order],
        ascending=False, na_position="last", kind="stable",
    )


**Notes:**

- Skor ditampilkan pada skala 0-100% (skor asli dikali 100).
- **HHEM Consistency** = Konsistensi fakta rangkuman terhadap teks sumber, dinilai oleh [Vectara HHEM-2.1-Open](https://huggingface.co/vectara/hallucination_evaluation_model). Lebih tinggi berarti lebih konsisten dengan sumber.
- **BERTScore F1** = Kemiripan makna rangkuman model dengan rangkuman referensi manusia, menggunakan [BERTScore](https://github.com/Tiiiger/bert_score). Lebih tinggi berarti lebih mirip.
- **Mean** = Rata-rata skor semua artikel dalam run yang dipilih. Mempertimbangkan setiap skor, tetapi lebih rentan terpengaruh nilai ekstrem (outlier).
- **Median** = Nilai tengah skor semua artikel dalam run yang dipilih. Lebih tahan terhadap outlier dan menggambarkan skor tipikal artikel.


## Speed dan kualitas per model

Model yang hanya memiliki hasil speed atau kualitas tetap ditampilkan. Data yang belum tersedia ditandai N/A.

In [51]:
speed_display_df = model_speed_df.rename(columns={
    "summarization_minutes": "Summarization Time",
    "prefill_tokens_per_second": "Prefill TPS",
    "generation_tokens_per_second": "Generation TPS",
})
speed_display_df.columns = pd.MultiIndex.from_product(
    [["Speed"], speed_display_df.columns], names=["Metric", "Statistic"],
)
quality_display_df = model_metrics_df.rename(columns={
    "hhem_consistency": "HHEM Consistency",
    "bertscore_f1": "BERTScore F1",
    "score_mean": "Mean",
    "score_median": "Median",
}).rename_axis(columns=["Metric", "Statistic"])
combined_df = pd.concat([speed_display_df, quality_display_df], axis=1).rename_axis("Model")
combined_df = combined_df.loc[~combined_df.index.astype(str).str.contains("luna", case=False, na=False)]

formats = {
    ("Speed", "Summarization Time"): "{:.0f} minutes",
    ("Speed", "Prefill TPS"): "{:.0f} tok/sec",
    ("Speed", "Generation TPS"): "{:.0f} tok/sec",
}
formats.update({column: "{:.2%}" for column in quality_display_df.columns})
display(combined_df.style.format(formats, na_rep="N/A"))
